In [1]:
# =============================================================================
# Load Libraries
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, QuantileTransformer, LabelEncoder
from sklearn.model_selection import StratifiedKFold  # Better for ROC
from sklearn.metrics import roc_auc_score           # Target metric
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# =============================================================================
# Configuration
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "ADD_EXTERN_DATA": False,
    "USE_STRATIFIED_SPLIT": True,
    "USE_EXTENDED_STRAT": False,
    "VAL_SIZE": 0.15,
    "N_FOLDS": 5,
    "SEEDS": [42, 137, 271, 401, 777],
    "GPU_ACC": True,
    "TARGET": "Heart Disease",
}

NUM_FEATURES = ["Age", "BP", "Cholesterol", "Max HR", "ST depression"]
CATS = [
    "Sex", "Chest pain type", "FBS over 120", "EKG results",
    "Exercise angina", "Slope of ST", "Number of vessels fluro", "Thallium"
]

print("Completed Loading Libraries and Configuration ...")

# =============================================================================
# Data Loading
# =============================================================================
train_df = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test_df = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

if CONFIG["ADD_EXTERN_DATA"]:
    original_df = pd.read_csv("/kaggle/input/heart-disease-original/Heart_Disease_Prediction.csv")
    train_df = pd.concat(
        [train_df.iloc[:, 1:], original_df[train_df.columns[1:]]],
        ignore_index=True
    ).reset_index().rename(columns={"index": "id"})

trainval = train_df.copy()
test = test_df.copy()

# Encode target
le = LabelEncoder()
trainval[CONFIG["TARGET"]] = le.fit_transform(trainval[CONFIG["TARGET"]]).astype(np.uint8)

print("Completed Loading Data ...")

Completed Loading Libraries and Configuration ...
Completed Loading Data ...


In [2]:
# =============================================================================
# Generate Summary
# =============================================================================

def create_summary(df):
    describe = df.describe().transpose()
    summary = pd.DataFrame(df.dtypes, columns=['DataType'])
    summary['MissingValues'] = df.isnull().sum()
    summary['UniqueValues'] = df.nunique()
    summary['FirstValue'] = df.iloc[0]
    summary['SecondValue'] = df.iloc[1]
    summary['ThirdValue'] = df.iloc[2]
    summary = pd.concat([summary, describe], axis=1)
    summary = summary.fillna('--')
    return summary

print("Summary Descfription of Training Dataset ...")
create_summary(train_df)

Summary Descfription of Training Dataset ...


,DataType,MissingValues,UniqueValues,FirstValue,SecondValue,ThirdValue,count,mean,std,min,25%,50%,75%,max
id,int64,0,630000,0,1,2,630000.0,314999.5,181865.479132,0.0,157499.75,314999.5,472499.25,629999.0
Age,int64,0,42,58,52,56,630000.0,54.136706,8.256301,29.0,48.0,54.0,60.0,77.0
Sex,int64,0,2,1,1,0,630000.0,0.714735,0.451541,0.0,0.0,1.0,1.0,1.0
Chest pain type,int64,0,4,4,1,2,630000.0,3.312752,0.851615,1.0,3.0,4.0,4.0,4.0
BP,int64,0,66,152,125,160,630000.0,130.497433,14.975802,94.0,120.0,130.0,140.0,200.0
Cholesterol,int64,0,150,239,325,188,630000.0,245.011814,33.681581,126.0,223.0,243.0,269.0,564.0
FBS over 120,int64,0,2,0,0,0,630000.0,0.079987,0.271274,0.0,0.0,0.0,0.0,1.0
EKG results,int64,0,3,0,2,2,630000.0,0.98166,0.998783,0.0,0.0,0.0,2.0,2.0
Max HR,int64,0,93,158,171,151,630000.0,152.816763,19.112927,71.0,142.0,157.0,166.0,202.0
Exercise angina,int64,0,2,1,0,0,630000.0,0.273725,0.44587,0.0,0.0,0.0,1.0,1.0


In [3]:
# =============================================================================
# Apply Feature Engineering
# =============================================================================
def add_engineered_features(df):
    df = df.copy()
    # Ratios (Adding epsilon 1e-6 is safer than +1 for zero-handling)
    df["chol_age_ratio"] = df["Cholesterol"] / (df["Age"] + 1)
    df["bp_age_ratio"]   = df["BP"] / (df["Age"] + 1)
    df["hr_age_ratio"]   = df["Max HR"] / (df["Age"] + 1)
    df["st_hr_ratio"]    = df["ST depression"] / (df["Max HR"] + 1)

    # Transforms
    df["age_sq"] = df["Age"] ** 2
    df["st_sq"]  = df["ST depression"] ** 2
    df["log_age"] = np.log1p(df["Age"])
    df["sqrt_chol"] = np.sqrt(df["Cholesterol"].clip(lower=0))

    # Binary Flags
    df["is_senior"]   = (df["Age"] >= 55).astype(np.uint8)
    df["high_chol"]   = (df["Cholesterol"] >= 240).astype(np.uint8)
    df["high_bp"]     = (df["BP"] >= 140).astype(np.uint8)
    df["low_hr"]      = (df["Max HR"] < 120).astype(np.uint8)
    df["st_present"]  = (df["ST depression"] > 0).astype(np.uint8)

    # Interactions (Converted to strings for easy concatenation)
    df["sex_chestpain"] = df["Sex"].astype(str) + "_" + df["Chest pain type"].astype(str)
    df["thal_vessels"]  = df["Thallium"].astype(str) + "_" + df["Number of vessels fluro"].astype(str)
    df["exercise_slope"] = df["Exercise angina"].astype(str) + "_" + df["Slope of ST"].astype(str)

    df['Age_x_MaxHR'] = df['Age'] * df['Max HR']
    df['Age_x_ST_depression'] = df['Age'] * df['ST depression']
    df['MaxHR_age_adjusted'] = df['Max HR'] - (220 - df['Age'])
    df['BP_age_risk'] = df['BP'] * (1 + (df['Age'] - 50) * 0.01)

    return df

train_df = add_engineered_features(train_df)
test_df = add_engineered_features(test_df)

# =============================================================================
# Define Final Column Lists
# =============================================================================

NUM_COLS_EXT = NUM_FEATURES + [
    "chol_age_ratio", "bp_age_ratio", "hr_age_ratio", "st_hr_ratio",
    "age_sq", "st_sq", "log_age", "sqrt_chol", "Age_x_MaxHR", 
    "Age_x_ST_depression", "MaxHR_age_adjusted", "BP_age_risk"
]

CAT_COLS_EXT = CATS + ["sex_chestpain", "thal_vessels", "exercise_slope"]

BIN_COLS = [
    "is_senior", "high_chol", "high_bp",
    "low_hr", "st_present"
]

# =============================================================================
# Encoding Target
# =============================================================================

le = LabelEncoder()
train_df[CONFIG["TARGET"]] = le.fit_transform(train_df[CONFIG["TARGET"]]).astype(np.uint8)


# =============================================================================
# Encoding New Categoricals
# =============================================================================

for col in CAT_COLS_EXT:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    train_df[col] = oe.fit_transform(train_df[[col]].astype(str)).astype(int)
    test_df[col] = oe.transform(test_df[[col]].astype(str)).astype(int)

print("Completed generating Feature Engineering ...")


Completed generating Feature Engineering ...


In [4]:
# =============================================================================
# Identify all numeric/binary columns to scale
# =============================================================================
TARGET = CONFIG["TARGET"]
SEED = CONFIG["SEEDS"][0]  # Taking the first seed from your list
all_num_cols = NUM_COLS_EXT + BIN_COLS

# =============================================================================
# Fit Scaler and Encoder
# =============================================================================

scaler = QuantileTransformer(output_distribution='normal', n_quantiles=1000, random_state=42)
scaler.fit(train_df[all_num_cols])

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder.fit(train_df[CAT_COLS_EXT])

def preprocess(df):
    nums = scaler.transform(df[all_num_cols]).astype(np.float32)
    
    cats = (encoder.transform(df[CAT_COLS_EXT]) + 1).astype(np.int64)
    return nums, cats

# =============================================================================
# Transform Data
# =============================================================================

X_num, X_cat = preprocess(train_df)
y = train_df[TARGET].values.astype(np.float32) # Standard for PyTorch Regression/BCELoss
X_test_num, X_test_cat = preprocess(test_df)

# =============================================================================
# Calculate Cardinalities for Embedding Layers
# =============================================================================

cat_cardinalities = [int(len(cat_list)) + 1 for cat_list in encoder.categories_]

print(f"Preprocessing completed. Numeric shape: {X_num.shape}, Categorical shape: {X_cat.shape}")
print(f"Embedding Cardinalities: {cat_cardinalities}")


Preprocessing completed. Numeric shape: (630000, 22), Categorical shape: (630000, 11)
Embedding Cardinalities: [3, 5, 3, 4, 3, 4, 5, 4, 9, 13, 7]


In [5]:
# =============================================================================
# --- PREPROCESSING ---
# =============================================================================

train_eng = add_engineered_features(train_df)
all_num_cols = [c for c in train_eng.columns if c not in CATS + [TARGET, 'id']]

scaler = QuantileTransformer(output_distribution='normal', n_quantiles=1000, random_state=SEED)
scaler.fit(train_eng[all_num_cols])

# Use unknowns = 0, so encoder starts at 1
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder.fit(train_eng[CATS])

def preprocess(df):
    df_eng = add_engineered_features(df)
    nums = scaler.transform(df_eng[all_num_cols])
    # Shift categories by +1 so that index 0 is reserved for unknowns
    cats = (encoder.transform(df_eng[CATS]) + 1).astype(np.int64)
    return nums, cats

X_num, X_cat = preprocess(train_df)
y = train_df[TARGET].values
X_test_num, X_test_cat = preprocess(test_df)

cat_cardinalities = [int(encoder.categories_[i].size) + 1 for i in range(len(CATS))]

print("Preprocessing completed")


Preprocessing completed


In [6]:
# =============================================================================
# Model Architecture
# =============================================================================
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.ln = nn.LayerNorm(dim)
        self.linear1 = nn.Linear(dim, dim * 2)
        self.linear2 = nn.Linear(dim * 2, dim)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, x):
        res = x
        out = self.ln(x)
        out = self.linear1(out)
        out = self.activation(out)
        out = self.linear2(out)
        out = self.dropout(out)
        return out + res

class TabularNet(nn.Module):
    def __init__(self, num_numerical, cat_cardinalities, embed_dim=12, hidden_dim=256):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(n, embed_dim) for n in cat_cardinalities
        ])
        total_cat_dim = len(cat_cardinalities) * embed_dim
        self.input_proj = nn.Linear(num_numerical + total_cat_dim, hidden_dim)
        
        self.res_layers = nn.ModuleList([ResidualBlock(hidden_dim) for _ in range(3)])
        
        # Binary Classification Head
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Linear(hidden_dim // 4, 1) # Outputs raw Logits
        )

    def forward(self, x_num, x_cat):
        x_embeds = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat([x_num] + x_embeds, dim=1)
        x = self.input_proj(x)
        for layer in self.res_layers:
            x = layer(x)
        return self.head(x).squeeze(1)
        

In [7]:
# =============================================================================
# Training Loop
# =============================================================================

all_seed_oof_preds = []
all_test_probs = []

test_num_tensor = torch.tensor(X_test_num, dtype=torch.float32).to(device)
test_cat_tensor = torch.tensor(X_test_cat, dtype=torch.int64).to(device)

for seed_idx, SEED in enumerate(CONFIG["SEEDS"]):
    print(f"\n{'='*20}")
    print(f" RUNNING SEED {SEED} ({seed_idx + 1}/{len(CONFIG['SEEDS'])})")
    print(f"{'='*20}")
    
    skf = StratifiedKFold(n_splits=CONFIG["N_FOLDS"], shuffle=True, random_state=SEED)
    seed_oof_preds = np.zeros(len(y))
    
    for fold, (t_idx, v_idx) in enumerate(skf.split(X_num, y)):
        print(f"Fold {fold+1}...")
        
        train_ds = TensorDataset(
            torch.tensor(X_num[t_idx], dtype=torch.float32), 
            torch.tensor(X_cat[t_idx], dtype=torch.int64), 
            torch.tensor(y[t_idx], dtype=torch.float32)
        )
        val_ds = TensorDataset(
            torch.tensor(X_num[v_idx], dtype=torch.float32), 
            torch.tensor(X_cat[v_idx], dtype=torch.int64), 
            torch.tensor(y[v_idx], dtype=torch.float32)
        )
        
        tl = DataLoader(train_ds, batch_size=1024, shuffle=True)
        vl = DataLoader(val_ds, batch_size=2048, shuffle=False)

        model = TabularNet(X_num.shape[1], cat_cardinalities).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        criterion = nn.BCEWithLogitsLoss()

        best_val_auc = 0
        patience = 5
        counter = 0

        for epoch in range(50):
            model.train()
            for xn, xc, yb in tl:
                xn, xc, yb = xn.to(device), xc.to(device), yb.to(device)
                optimizer.zero_grad()
                logits = model(xn, xc)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()
            
            scheduler.step()
            
            # Validation
            model.eval()
            p_list = []
            with torch.no_grad():
                for xn, xc, yb in vl:
                    probs = torch.sigmoid(model(xn.to(device), xc.to(device)))
                    p_list.append(probs.cpu().numpy())
            
            val_probs = np.concatenate(p_list)
            cur_auc = roc_auc_score(y[v_idx], val_probs)
            
            if cur_auc > best_val_auc:
                best_val_auc = cur_auc
                # Save with unique name per seed/fold
                torch.save(model.state_dict(), f"model_s{SEED}_f{fold}.pth")
                counter = 0
            else:
                counter += 1
                if counter >= patience: break
        
        # Load best and predict OOF
        model.load_state_dict(torch.load(f"model_s{SEED}_f{fold}.pth"))
        model.eval()
        
        # OOF Predictions
        p_list = []
        with torch.no_grad():
            for xn, xc, _ in vl:
                p_list.append(torch.sigmoid(model(xn.to(device), xc.to(device))).cpu().numpy())
        seed_oof_preds[v_idx] = np.concatenate(p_list)
        
        # Test Inference
        with torch.no_grad():
            test_logits = model(test_num_tensor, test_cat_tensor)
            all_test_probs.append(torch.sigmoid(test_logits).cpu().numpy())

    all_seed_oof_preds.append(seed_oof_preds)
    print(f"SEED {SEED} OOF AUC: {roc_auc_score(y, seed_oof_preds):.5f}")
    


 RUNNING SEED 42 (1/5)
Fold 1...
Fold 2...
Fold 3...
Fold 4...
Fold 5...
SEED 42 OOF AUC: 0.95331

 RUNNING SEED 137 (2/5)
Fold 1...
Fold 2...
Fold 3...
Fold 4...
Fold 5...
SEED 137 OOF AUC: 0.95328

 RUNNING SEED 271 (3/5)
Fold 1...
Fold 2...
Fold 3...
Fold 4...
Fold 5...
SEED 271 OOF AUC: 0.95333

 RUNNING SEED 401 (4/5)
Fold 1...
Fold 2...
Fold 3...
Fold 4...
Fold 5...
SEED 401 OOF AUC: 0.95331

 RUNNING SEED 777 (5/5)
Fold 1...
Fold 2...
Fold 3...
Fold 4...
Fold 5...
SEED 777 OOF AUC: 0.95315


In [8]:
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

y_array = np.asarray(y).ravel()

# ============================================================
# Calculate Seed Weights Based on OOF Performance
# ============================================================

seed_scores = [roc_auc_score(y_array, np.asarray(oof).ravel())
               for oof in all_seed_oof_preds]

raw_weights = np.array(seed_scores) - 0.5

if raw_weights.sum() <= 0:
    print("Warning: Non-positive weight sum detected. Using equal weights.")
    normalized_weights = np.ones_like(raw_weights) / len(raw_weights)
else:
    normalized_weights = raw_weights / raw_weights.sum()

print("\n--- Seed Weights based on Performance ---")
for i, s in enumerate(CONFIG["SEEDS"]):
    print(f"Seed {s}: Weight {normalized_weights[i]:.4f} (AUC: {seed_scores[i]:.5f})")

# ============================================================
# Weighted Rank Averaging for OOF
# ============================================================

weighted_ranked_oof = np.zeros(len(y_array), dtype=float)

for i, oof in enumerate(all_seed_oof_preds):
    ranked = rankdata(oof) / len(oof)   # normalize ranks to [0,1]
    weighted_ranked_oof += ranked * normalized_weights[i]

# ============================================================
# Weighted Rank Averaging for Test
# ============================================================

weighted_ranked_test = np.zeros(len(test_df), dtype=float)
num_folds = CONFIG["N_FOLDS"]

for i, seed_weight in enumerate(normalized_weights):

    seed_test_probs = all_test_probs[i * num_folds : (i + 1) * num_folds]

    # Average ranks within seed first
    seed_rank_sum = np.zeros(len(test_df), dtype=float)

    for fold_prob in seed_test_probs:
        seed_rank_sum += rankdata(fold_prob) / len(fold_prob)

    seed_rank_avg = seed_rank_sum / num_folds

    weighted_ranked_test += seed_rank_avg * seed_weight

# ============================================================
# Final Normalization to [0,1]
# ============================================================

min_val = weighted_ranked_test.min()
max_val = weighted_ranked_test.max()

if max_val - min_val == 0:
    print("Warning: Zero variance in test predictions.")
    final_test_output = np.full_like(weighted_ranked_test, 0.5)
else:
    final_test_output = (weighted_ranked_test - min_val) / (max_val - min_val)

# ============================================================
# Final OOF AUC
# ============================================================

final_oof_auc = roc_auc_score(y_array, weighted_ranked_oof)

print(f"\n{'*'*40}")
print(f"Final Weighted Ranked OOF AUC: {final_oof_auc:.5f}")
print(f"{'*'*40}")

# ============================================================
# Save Submission
# ============================================================

submission = pd.DataFrame({
    'id': test_df['id'],
    CONFIG['TARGET']: final_test_output
})

submission.to_csv("submission.csv", index=False)

print("\nSuccess! Submission saved with Weighted Rank Averaging.")

submission.head(20)


--- Seed Weights based on Performance ---
Seed 42: Weight 0.2000 (AUC: 0.95331)
Seed 137: Weight 0.2000 (AUC: 0.95328)
Seed 271: Weight 0.2000 (AUC: 0.95333)
Seed 401: Weight 0.2000 (AUC: 0.95331)
Seed 777: Weight 0.1999 (AUC: 0.95315)

****************************************
Final Weighted Ranked OOF AUC: 0.95349
****************************************

Success! Submission saved with Weighted Rank Averaging.


,id,Heart Disease
0,630000,0.765128
1,630001,0.040747
2,630002,0.901160
3,630003,0.022982
4,630004,0.492180
5,630005,0.893372
6,630006,0.111052
7,630007,0.621160
8,630008,0.950184
9,630009,0.084273
